In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
# %%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
#from tpch import load_lineitem, load_part, q19
import pandas as pd
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [5]:

### cell 1 ###

def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [6]:
# %%time
# ### cell 2 ###

# SM_SMALL = ["SM BOX", "SM CASE", "SM PACK", "SM PKG"]
# MED_CONTAINER = ["MED BAG", "MED BOX", "MED PACK", "MED PKG"]
# LG_CONTAINER = ["LG BOX", "LG CASE", "LG PACK", "LG PKG"]
# SHIPMODES = ["AIR", "AIRREG"]

# # Precompute lineitem columns to cut down __getitem__ calls
# lq_line = lineitem["L_QUANTITY"]
# si_line = lineitem["L_SHIPINSTRUCT"]
# sm_line = lineitem["L_SHIPMODE"]

# fline = lineitem[
#     lq_line.between(4, 36)
#     & (si_line == "DELIVER IN PERSON")
#     & sm_line.isin(SHIPMODES)
# ]

# # Precompute part columns
# pb_part = part["P_BRAND"]
# pc_part = part["P_CONTAINER"]
# ps_part = part["P_SIZE"]

# fpart = part[
#     ((pb_part == "Brand#31") & pc_part.isin(SM_SMALL) & ps_part.between(1, 5))
#     | ((pb_part == "Brand#43") & pc_part.isin(MED_CONTAINER) & ps_part.between(1, 10))
#     | ((pb_part == "Brand#43") & pc_part.isin(LG_CONTAINER) & ps_part.between(1, 15))
# ]

# # Join
# df = fline.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

# # Precompute merged columns
# lq = df["L_QUANTITY"]
# pb = df["P_BRAND"]
# pc = df["P_CONTAINER"]

# # Final filter (size checks for Brand#31 guaranteed by fpart)
# mask_brand31 = (pb == "Brand#31") & lq.between(4, 14)
# mask_brand43 = (pb == "Brand#43") & (
#     (pc.isin(MED_CONTAINER) & lq.between(15, 25))
#     | (pc.isin(LG_CONTAINER)  & lq.between(26, 36))
# )
# df = df[mask_brand31 | mask_brand43]

# # Compute total on GPU
# total = (df["L_EXTENDEDPRICE"] * (1.0 - df["L_DISCOUNT"])) .sum()

In [7]:
# %%time
# ### cell 2 ###

# SM_SMALL = ["SM BOX", "SM CASE", "SM PACK", "SM PKG"]
# MED_CONTAINER = ["MED BAG", "MED BOX", "MED PACK", "MED PKG"]
# LG_CONTAINER = ["LG BOX", "LG CASE", "LG PACK", "LG PKG"]
# SHIPMODES = ["AIR", "AIRREG"]

# # Precompute lineitem columns to cut down __getitem__ calls
# lq_line = lineitem["L_QUANTITY"]
# si_line = lineitem["L_SHIPINSTRUCT"]
# sm_line = lineitem["L_SHIPMODE"]

# fline = lineitem[
#     lq_line.between(4, 36)
#     & (si_line == "DELIVER IN PERSON")
#     & sm_line.isin(SHIPMODES)
# ]

# # Precompute part columns
# pb_part = part["P_BRAND"]
# pc_part = part["P_CONTAINER"]
# ps_part = part["P_SIZE"]

# fpart = part[
#     ((pb_part == "Brand#31") & pc_part.isin(SM_SMALL) & ps_part.between(1, 5))
#     | ((pb_part == "Brand#43") & pc_part.isin(MED_CONTAINER) & ps_part.between(1, 10))
#     | ((pb_part == "Brand#43") & pc_part.isin(LG_CONTAINER) & ps_part.between(1, 15))
# ]

# # Join
# df = fline.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

# # Precompute merged columns
# lq = df["L_QUANTITY"]
# pb = df["P_BRAND"]
# pc = df["P_CONTAINER"]

# # Final filter (size checks for Brand#31 guaranteed by fpart)
# mask_brand31 = (pb == "Brand#31") & lq.between(4, 14)
# mask_brand43 = (pb == "Brand#43") & (
#     (pc.isin(MED_CONTAINER) & lq.between(15, 25))
#     | (pc.isin(LG_CONTAINER)  & lq.between(26, 36))
# )
# df = df[mask_brand31 | mask_brand43]

# # Compute total on GPU
# total = (df["L_EXTENDEDPRICE"] * (1.0 - df["L_DISCOUNT"])) .sum()

In [8]:
# total

In [9]:
%%time
Brand31 = "Brand#31"
Brand43 = "Brand#43"
SMBOX = "SM BOX"
SMCASE = "SM CASE"
SMPACK = "SM PACK"
SMPKG = "SM PKG"
MEDBAG = "MED BAG"
MEDBOX = "MED BOX"
MEDPACK = "MED PACK"
MEDPKG = "MED PKG"
LGBOX = "LG BOX"
LGCASE = "LG CASE"
LGPACK = "LG PACK"
LGPKG = "LG PKG"
DELIVERINPERSON = "DELIVER IN PERSON"
AIR = "AIR"
AIRREG = "AIRREG"
lsel = (
    (
        ((lineitem.L_QUANTITY <= 36) & (lineitem.L_QUANTITY >= 26))
        | ((lineitem.L_QUANTITY <= 25) & (lineitem.L_QUANTITY >= 15))
        | ((lineitem.L_QUANTITY <= 14) & (lineitem.L_QUANTITY >= 4))
    )
    & (lineitem.L_SHIPINSTRUCT == DELIVERINPERSON)
    & ((lineitem.L_SHIPMODE == AIR) | (lineitem.L_SHIPMODE == AIRREG))
)
psel = (part.P_SIZE >= 1) & (
    (
        (part.P_SIZE <= 5)
        & (part.P_BRAND == Brand31)
        & (
            (part.P_CONTAINER == SMBOX)
            | (part.P_CONTAINER == SMCASE)
            | (part.P_CONTAINER == SMPACK)
            | (part.P_CONTAINER == SMPKG)
        )
    )
    | (
        (part.P_SIZE <= 10)
        & (part.P_BRAND == Brand43)
        & (
            (part.P_CONTAINER == MEDBAG)
            | (part.P_CONTAINER == MEDBOX)
            | (part.P_CONTAINER == MEDPACK)
            | (part.P_CONTAINER == MEDPKG)
        )
    )
    | (
        (part.P_SIZE <= 15)
        & (part.P_BRAND == Brand43)
        & (
            (part.P_CONTAINER == LGBOX)
            | (part.P_CONTAINER == LGCASE)
            | (part.P_CONTAINER == LGPACK)
            | (part.P_CONTAINER == LGPKG)
        )
    )
)
flineitem = lineitem[lsel]
fpart = part[psel]
jn = flineitem.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")
jnsel = (
    (jn.P_BRAND == Brand31)
    & (
        (jn.P_CONTAINER == SMBOX)
        | (jn.P_CONTAINER == SMCASE)
        | (jn.P_CONTAINER == SMPACK)
        | (jn.P_CONTAINER == SMPKG)
    )
    & (jn.L_QUANTITY >= 4)
    & (jn.L_QUANTITY <= 14)
    & (jn.P_SIZE <= 5)
    | (jn.P_BRAND == Brand43)
    & (
        (jn.P_CONTAINER == MEDBAG)
        | (jn.P_CONTAINER == MEDBOX)
        | (jn.P_CONTAINER == MEDPACK)
        | (jn.P_CONTAINER == MEDPKG)
    )
    & (jn.L_QUANTITY >= 15)
    & (jn.L_QUANTITY <= 25)
    & (jn.P_SIZE <= 10)
    | (jn.P_BRAND == Brand43)
    & (
        (jn.P_CONTAINER == LGBOX)
        | (jn.P_CONTAINER == LGCASE)
        | (jn.P_CONTAINER == LGPACK)
        | (jn.P_CONTAINER == LGPKG)
    )
    & (jn.L_QUANTITY >= 26)
    & (jn.L_QUANTITY <= 36)
    & (jn.P_SIZE <= 15)
)
jn = jn[jnsel]
total = (jn.L_EXTENDEDPRICE * (1.0 - jn.L_DISCOUNT)).sum()
    

CPU times: user 1.37 s, sys: 462 ms, total: 1.84 s
Wall time: 1.6 s


In [8]:
total

np.float64(38457884.1075)

In [9]:
### cell 3 ###

total = total + 1